# 02 — Baseline Custom CNN

Train a 4-block CNN from scratch as our baseline model.

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath(".."))

# CUDA setup — must be before TF import
os.environ["XLA_FLAGS"] = "--xla_gpu_cuda_data_dir=/home/yuujin/school/cv-final-project/.cuda"

import tensorflow as tf
print(f"TensorFlow {tf.__version__}")
print(f"GPUs available: {tf.config.list_physical_devices('GPU')}")

# Enable GPU memory growth to avoid allocating all VRAM at once
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

In [ ]:
from src.data_loader import load_and_split_data, create_dataset, get_class_weights
from src.models import build_custom_cnn
from src.train import get_callbacks, train_model
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_training_curves

DATA_DIR = os.path.join("..", "data", "raw", "standardized_256")

## 1. Load and Split Data

In [ ]:
train_paths, train_labels, val_paths, val_labels, test_paths, test_labels, class_names = \
    load_and_split_data(DATA_DIR)

print(f"Classes: {class_names}")
print(f"Train: {len(train_paths)} | Val: {len(val_paths)} | Test: {len(test_paths)}")

In [ ]:
train_ds = create_dataset(train_paths, train_labels, augment=True)
val_ds = create_dataset(val_paths, val_labels, augment=False, shuffle=False)
test_ds = create_dataset(test_paths, test_labels, augment=False, shuffle=False)

class_weights = get_class_weights(train_labels)
print(f"Class weights: {class_weights}")

## 2. Build Model

In [ ]:
num_classes = len(class_names)
model = build_custom_cnn(num_classes)
model.summary()

## 3. Train

In [ ]:
callbacks = get_callbacks("custom_cnn")
history = train_model(
    model, train_ds, val_ds,
    epochs=50,
    class_weights=class_weights,
    callbacks=callbacks,
    lr=1e-3,
)

## 4. Training Curves

In [ ]:
plot_training_curves(
    history.history,
    title="Custom CNN",
    save_path=os.path.join("..", "outputs", "plots", "custom_cnn_curves.png"),
)

## 5. Evaluate on Test Set

In [ ]:
results = evaluate_model(model, test_ds, class_names)
print(f"Test Accuracy: {results['accuracy']:.4f}")
print("\nClassification Report:")
print(results["report"])

In [ ]:
plot_confusion_matrix(
    results["confusion_matrix"],
    class_names,
    title="Custom CNN — Confusion Matrix",
    save_path=os.path.join("..", "outputs", "plots", "custom_cnn_cm.png"),
)